# Simultaneous Inference — Batched Label-Logit + Noisy-OR

Runs all three adapters in ONE batched forward pass (per-sample `adapter_names`) instead of a
sequential `set_adapter` loop. Built on the verified faithful setup: Unsloth base, row's own
`formatted_text`, per-adapter label ids.

**Batching + the branch-point trick.** The sequential scorer steps forward a variable number of
tokens to reach the label. To batch, we instead append the FIXED formatting tokens the model emits
before the label (`\n` then indent whitespace — ids 107, 144 from the diagnostics) so every row's
label lands at the same final position in one pass. A verification cell confirms the batched
probabilities match the proven sequential scorer before we trust them.

Run on Kaggle **GPU T4**. Set `HF_TOKEN` as a Kaggle secret (do NOT hardcode it).

In [ ]:
try:
    import unsloth
except ImportError:
    !pip install -q unsloth
from unsloth import FastModel
import torch, pandas as pd
from datasets import load_dataset
from huggingface_hub import login
from peft import PeftModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


In [ ]:
HF_TOKEN = ""
login(token=HF_TOKEN)

HF_USERNAME = "hirushafernando"
BASE_MODEL  = "unsloth/gemma-3-1b-it-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048

ADAPTERS = {
    "role_violation":       f"{HF_USERNAME}/slm-shield-role-and-instruction-violation-qlora",
    "privilege_escalation": f"{HF_USERNAME}/slm-shield-privilege-escalation-qlora",
    "obfuscation":          f"{HF_USERNAME}/slm-shield-obfuscation-and-evation-patterns-qlora",
}
DATASETS = {
    "role_violation":       f"{HF_USERNAME}/fyp-slm-a",
    "privilege_escalation": f"{HF_USERNAME}/fyp-slm-b",
    "obfuscation":          f"{HF_USERNAME}/fyp-slm-c",
}
ADAPTER_ORDER = ["role_violation", "privilege_escalation", "obfuscation"]

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = PeftModel.from_pretrained(model, ADAPTERS["role_violation"], adapter_name="role_violation")
model.load_adapter(ADAPTERS["privilege_escalation"], adapter_name="privilege_escalation")
model.load_adapter(ADAPTERS["obfuscation"], adapter_name="obfuscation")
FastModel.for_inference(model)
print("adapters:", list(model.peft_config.keys()))

In [ ]:
MARKER = "<start_of_turn>model"

def get_prompt_without_answer(formatted_text: str) -> str:
    if formatted_text.startswith("<bos>"):
        formatted_text = formatted_text[len("<bos>"):]
    if MARKER not in formatted_text:
        raise ValueError("no model-turn marker")
    return formatted_text.split(MARKER)[0] + MARKER

def get_gold_label_word(formatted_text: str) -> str:
    after = formatted_text.split(MARKER)[1]
    return after.replace("<end_of_turn>", "").strip().split()[0].upper()

In [ ]:
# One benign + one attack row per dataset (for label-id derivation and verification)
rows = {}
for a in ADAPTER_ORDER:
    ds = load_dataset(DATASETS[a], token=HF_TOKEN)["validation"]
    rows[a] = {"inj": next(r for r in ds if r["label"] == 1),
               "ben": next(r for r in ds if r["label"] == 0)}
    print(f"[{a}] label1={get_gold_label_word(rows[a]['inj']['formatted_text'])!r} "
          f"label0={get_gold_label_word(rows[a]['ben']['formatted_text'])!r}")

In [ ]:
# Derive label ids: first NON-whitespace token after the model marker (skips '\n' and indent).
def find_label_position(dec, marker_word="model"):
    mi = max(k for k, d in enumerate(dec) if d.strip() == marker_word)
    j = mi + 1
    while j < len(dec) and dec[j].strip() == "":
        j += 1
    return j

def derive_label_ids(ft_inj, ft_ben):
    def toks(ft):
        full = ft[len("<bos>"):] if ft.startswith("<bos>") else ft
        ids = tokenizer(full, add_special_tokens=True)["input_ids"]
        return ids, [tokenizer.decode([i]) for i in ids]
    ii, di = toks(ft_inj); ib, db = toks(ft_ben)
    pi, pb = find_label_position(di), find_label_position(db)
    return ii[pi], ib[pb], di[pi], db[pb]

LABEL_IDS = {}
for a in ADAPTER_ORDER:
    inj_id, ben_id, it, bt = derive_label_ids(rows[a]["inj"]["formatted_text"], rows[a]["ben"]["formatted_text"])
    LABEL_IDS[a] = {"inj": inj_id, "ben": ben_id}
    print(f"[{a:22s}] INJ={inj_id}({it!r}) BEN={ben_id}({bt!r}) distinct={inj_id!=ben_id}")
assert all(v["inj"] != v["ben"] for v in LABEL_IDS.values()), "label ids collide"
# NOTE: all three use INJ=1204 ('IN'), BEN=88980 ('BEN') — same ids across adapters is expected.
INJ_ID = LABEL_IDS[ADAPTER_ORDER[0]]["inj"]
BEN_ID = LABEL_IDS[ADAPTER_ORDER[0]]["ben"]

In [ ]:
# Derive the FIXED formatting tokens between the marker and the label, from a real row.
# (Diagnostics showed: <...model> then '\n'(107) then '        '(144) then 'IN'(1204).)
def marker_to_label_suffix(formatted_text):
    full = formatted_text[len("<bos>"):] if formatted_text.startswith("<bos>") else formatted_text
    ids = tokenizer(full, add_special_tokens=True)["input_ids"]
    dec = [tokenizer.decode([i]) for i in ids]
    mi = max(k for k, d in enumerate(dec) if d.strip() == "model")
    lp = mi + 1
    while dec[lp].strip() == "":
        lp += 1
    return ids[mi+1:lp]                 # the formatting tokens strictly between marker and label

SUFFIX_IDS = marker_to_label_suffix(rows[ADAPTER_ORDER[0]]["inj"]["formatted_text"])
print("formatting tokens between marker and label:", SUFFIX_IDS,
      [tokenizer.decode([i]) for i in SUFFIX_IDS])

## Sequential reference scorer (the proven oracle)

In [ ]:
@torch.inference_mode()
def score_row_sequential(formatted_text, adapter, max_pre=6):
    model.set_adapter(adapter)
    prompt = get_prompt_without_answer(formatted_text)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)["input_ids"]
    inj, ben = LABEL_IDS[adapter]["inj"], LABEL_IDS[adapter]["ben"]
    for _ in range(max_pre):
        logits = model(input_ids=ids).logits[0, -1, :]
        nxt = int(logits.argmax())
        if nxt in (inj, ben):
            return torch.softmax(torch.stack([logits[inj], logits[ben]]).float(), -1)[0].item()
        ids = torch.cat([ids, torch.tensor([[nxt]], device=ids.device)], dim=1)
    logits = model(input_ids=ids).logits[0, -1, :]
    return torch.softmax(torch.stack([logits[inj], logits[ben]]).float(), -1)[0].item()

## Batched simultaneous scorer — all adapters in ONE forward pass
Builds each adapter's prompt ending exactly at the label position (marker + formatting tokens),
left-pads the batch, and reads the label logits at the final position with per-sample adapters.

In [ ]:
@torch.inference_mode()
def score_all_batched(formatted_text_by_adapter):
    """formatted_text_by_adapter: {adapter: that adapter's formatted_text (or reconstructed prompt)}.
    Returns {adapter: p_injection} from ONE batched forward pass."""
    # Build each row's ids ending at the label branch point: prompt-without-answer + SUFFIX_IDS.
    seqs = []
    for a in ADAPTER_ORDER:
        prompt = get_prompt_without_answer(formatted_text_by_adapter[a])
        base = tokenizer(prompt, add_special_tokens=True)["input_ids"]
        seqs.append(base + SUFFIX_IDS)               # now the NEXT token is the label

    maxlen = max(len(s) for s in seqs)
    pad_id = tokenizer.pad_token_id
    input_ids, attn = [], []
    for s in seqs:
        padn = maxlen - len(s)
        input_ids.append([pad_id]*padn + s)          # LEFT pad -> real token at position -1
        attn.append([0]*padn + [1]*len(s))
    input_ids = torch.tensor(input_ids, device=model.device)
    attn = torch.tensor(attn, device=model.device)

    try:
        logits = model(input_ids=input_ids, attention_mask=attn,
                       adapter_names=list(ADAPTER_ORDER)).logits[:, -1, :]
    except TypeError:
        print("adapter_names unsupported -> sequential fallback")
        return {a: score_row_sequential(formatted_text_by_adapter[a], a) for a in ADAPTER_ORDER}

    out = {}
    for i, a in enumerate(ADAPTER_ORDER):
        inj, ben = LABEL_IDS[a]["inj"], LABEL_IDS[a]["ben"]
        pair = torch.stack([logits[i, inj], logits[i, ben]]).float()
        out[a] = torch.softmax(pair, -1)[0].item()
    return out

## Verification gate — batched must match sequential (on real rows)

In [ ]:
# Use each adapter's own attack/benign rows as the per-adapter inputs.
def check(kind, tol=2e-2):
    fbya = {a: rows[a][kind]["formatted_text"] for a in ADAPTER_ORDER}
    bat = score_all_batched(fbya)
    seq = {a: score_row_sequential(fbya[a], a) for a in ADAPTER_ORDER}
    md = 0.0
    for a in ADAPTER_ORDER:
        d = abs(bat[a]-seq[a]); md = max(md, d)
        print(f"[{a:22s}] {kind}: batched={bat[a]:.3f} sequential={seq[a]:.3f} diff={d:.2e}")
    print(f"  max diff={md:.2e} (tol {tol:.0e})\n")
    return md

m1 = check("inj"); m2 = check("ben")
assert max(m1, m2) < 2e-2, "batched != sequential; investigate padding/adapter_names"
print("PASS: batched simultaneous scoring matches sequential.")

## 1. Build the unified splits (dedup, category-tagged)

In [ ]:
import hashlib, os, json
import numpy as np, pandas as pd
from datasets import load_dataset

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

def row_hash(ft):
    return hashlib.sha1(ft.encode()).hexdigest()[:16]

def build_union(split):
    recs = []
    for a in ADAPTER_ORDER:
        ds = load_dataset(DATASETS[a], token=HF_TOKEN)[split]
        for r in ds:
            recs.append({"hash": row_hash(r["formatted_text"]),
                         "formatted_text": r["formatted_text"],
                         "label": int(r["label"]),
                         "source": a})
    df = pd.DataFrame(recs).drop_duplicates("hash").reset_index(drop=True)
    return df

val_df  = build_union("validation")
test_df = build_union("test")
# guard: no overlap between val and test
overlap = set(val_df["hash"]) & set(test_df["hash"])
print(f"val={len(val_df)} test={len(test_df)} overlap={len(overlap)}")
if overlap:
    test_df = test_df[~test_df["hash"].isin(overlap)].reset_index(drop=True)
    print(f"dropped {len(overlap)} overlapping rows from test -> test={len(test_df)}")
for name, df in [("val", val_df), ("test", test_df)]:
    print(name, "label balance:", df["label"].value_counts().to_dict(),
          "| by source:", df["source"].value_counts().to_dict())

## 2. Bulk score (resumable). Each row scored by ALL THREE adapters in one batched pass.

In [ ]:
def bulk_score(df, out_path, log_every=200):
    # resume if partial file exists
    if os.path.exists(out_path):
        done = pd.read_parquet(out_path)
        done_hashes = set(done["hash"])
        print(f"resuming: {len(done_hashes)} already scored")
    else:
        done = pd.DataFrame()
        done_hashes = set()
    buf = []
    todo = df[~df["hash"].isin(done_hashes)].reset_index(drop=True)
    for i, r in todo.iterrows():
        ft = r["formatted_text"]
        # every adapter scores THIS row's own formatted_text (faithful eval path)
        ps = score_all_batched({a: ft for a in ADAPTER_ORDER})
        buf.append({"hash": r["hash"], "label": r["label"], "source": r["source"],
                    **{f"p_{a}": ps[a] for a in ADAPTER_ORDER}})
        if (i+1) % log_every == 0:
            partial = pd.concat([done, pd.DataFrame(buf)], ignore_index=True)
            partial.to_parquet(out_path)
            print(f"  {i+1}/{len(todo)} scored, checkpointed")
    final = pd.concat([done, pd.DataFrame(buf)], ignore_index=True)
    final.to_parquet(out_path)
    print(f"done: {len(final)} rows -> {out_path}")
    return final

val_scores  = bulk_score(val_df,  f"{OUT}/val_scores.parquet")
test_scores = bulk_score(test_df, f"{OUT}/test_scores.parquet")

## 3. Fit noisy-OR q and tau on VALIDATION (CPU, seconds)

In [ ]:
from scipy.optimize import minimize
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score,
                             confusion_matrix, f1_score)

def P_matrix(df):
    return df[[f"p_{a}" for a in ADAPTER_ORDER]].values

def fused_S(q, P):
    return 1 - np.prod(1 - q * P, axis=1)

def bce(raw_q, P, y):
    q = 1/(1+np.exp(-raw_q))
    S = np.clip(fused_S(q, P), 1e-7, 1-1e-7)
    return -np.mean(y*np.log(S) + (1-y)*np.log(1-S))

Pv, yv = P_matrix(val_scores), val_scores["label"].values
res = minimize(bce, np.zeros(3), args=(Pv, yv), method="L-BFGS-B")
learned_q = 1/(1+np.exp(-res.x))
Q = dict(zip(ADAPTER_ORDER, learned_q))
print("learned reliabilities q:", {a: round(float(v),3) for a,v in Q.items()})
print("(higher q = adapter more trusted; expect role_violation lower if it is noisier)")

In [ ]:
# Pick tau on validation. Two options; report both.
def best_tau_f1(S, y):
    ts = np.unique(S)
    return max(ts, key=lambda t: f1_score(y, (S > t).astype(int), zero_division=0))

def tau_at_fpr(S, y, target_fpr=0.01):
    # threshold so benign FPR <= target on validation
    benign = np.sort(S[y == 0])
    k = int(np.ceil((1 - target_fpr) * (len(benign) + 1))) - 1
    k = min(max(k, 0), len(benign) - 1)
    return benign[k]

S_val_learned = fused_S(learned_q, Pv)
S_val_or      = fused_S(np.ones(3), Pv)      # probabilistic-OR baseline

TAU_learned_f1  = best_tau_f1(S_val_learned, yv)
TAU_learned_fpr = tau_at_fpr(S_val_learned, yv, 0.01)
TAU_or_f1       = best_tau_f1(S_val_or, yv)
print(f"noisy-OR   tau (max-F1)={TAU_learned_f1:.4f}  tau (1%FPR)={TAU_learned_fpr:.4f}")
print(f"prob-OR    tau (max-F1)={TAU_or_f1:.4f}")

## 4. Evaluate on TEST (read once). Fused metrics + per-adapter + baseline.

In [ ]:
Pt, yt = P_matrix(test_scores), test_scores["label"].values

def metrics(y, S, tau):
    pred = (S > tau).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="binary", pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    fpr = fp/(fp+tn) if (fp+tn) else 0.0
    try: auroc = roc_auc_score(y, S)
    except Exception: auroc = float("nan")
    return {"precision": round(p,4), "recall": round(r,4), "f1": round(f1,4),
            "fpr": round(fpr,4), "auroc": round(auroc,4), "tp": int(tp), "fp": int(fp),
            "fn": int(fn), "tn": int(tn)}

def tpr_at_fpr(y, S, target=0.01):
    order = np.argsort(-S)
    ys = y[order]
    n_neg = (y==0).sum(); n_pos = (y==1).sum()
    fp = tp = 0
    for lab in ys:
        if lab==1: tp += 1
        else: fp += 1
        if fp/n_neg > target:
            break
    return tp/n_pos

S_test_learned = fused_S(learned_q, Pt)
S_test_or      = fused_S(np.ones(3), Pt)

print("=== FUSED RESULTS ON TEST ===")
print("noisy-OR (learned q), tau@maxF1 :", metrics(yt, S_test_learned, TAU_learned_f1))
print("noisy-OR (learned q), tau@1%FPR :", metrics(yt, S_test_learned, TAU_learned_fpr))
print("prob-OR  (q=1)        tau@maxF1 :", metrics(yt, S_test_or, TAU_or_f1))
print()
print("TPR@1%FPR  noisy-OR:", round(tpr_at_fpr(yt, S_test_learned),4),
      " prob-OR:", round(tpr_at_fpr(yt, S_test_or),4))

In [ ]:
# Per-adapter metrics on test (each adapter alone at 0.5), + per-source recall breakdown.
print("=== PER-ADAPTER (threshold 0.5) ===")
for i, a in enumerate(ADAPTER_ORDER):
    print(f"{a:22s}:", metrics(yt, Pt[:, i], 0.5))
print()
print("=== FUSED noisy-OR recall by attack source (injection rows only) ===")
pred = (S_test_learned > TAU_learned_f1).astype(int)
tdf = test_scores.copy(); tdf["pred"] = pred
for a in ADAPTER_ORDER:
    sub = tdf[(tdf["source"]==a) & (tdf["label"]==1)]
    if len(sub):
        print(f"  {a:22s}: recall={sub['pred'].mean():.4f}  (n={len(sub)})")

## 5. (Optional) Merged baseline adapter — load, score test, compare
Load the already-trained merged adapter and score the SAME test rows for config (a).
Requires the merged adapter's own instruction template; it uses the same label ids.

In [ ]:
# Attach the merged adapter and score test rows with it (single adapter, no fusion).
MERGED_REPO = f"{HF_USERNAME}/fyp-gemma3-1b-slm-merged-qlora"   # adjust if your repo name differs
try:
    model.load_adapter(MERGED_REPO, adapter_name="merged")
    print("merged adapter loaded")
    MERGED_OK = True
except Exception as e:
    print("could not load merged adapter:", e); MERGED_OK = False

if MERGED_OK:
    import torch
    @torch.inference_mode()
    def score_merged(formatted_text, max_pre=6):
        model.set_adapter("merged")
        prompt = get_prompt_without_answer(formatted_text)
        ids = tokenizer(prompt, return_tensors="pt").to(model.device)["input_ids"]
        for _ in range(max_pre):
            logits = model(input_ids=ids).logits[0, -1, :]
            nxt = int(logits.argmax())
            if nxt in (INJ_ID, BEN_ID):
                return torch.softmax(torch.stack([logits[INJ_ID], logits[BEN_ID]]).float(), -1)[0].item()
            ids = torch.cat([ids, torch.tensor([[nxt]], device=ids.device)], dim=1)
        logits = model(input_ids=ids).logits[0, -1, :]
        return torch.softmax(torch.stack([logits[INJ_ID], logits[BEN_ID]]).float(), -1)[0].item()

    # need the raw formatted_text for test rows: join scores back to test_df by hash
    ft_by_hash = dict(zip(test_df["hash"], test_df["formatted_text"]))
    S_merged = np.array([score_merged(ft_by_hash[h]) for h in test_scores["hash"]])
    tau_m = best_tau_f1(fused_S(np.ones(3), Pv), yv)  # not used; pick merged tau on val separately
    # fit merged tau on a val scoring would be ideal; for a quick compare use 0.5
    print("merged baseline (tau=0.5):", metrics(yt, S_merged, 0.5))
    print("merged TPR@1%FPR:", round(tpr_at_fpr(yt, S_merged),4))

## 6. Save everything for the thesis
Persist scores + fitted params + metrics so nothing depends on the live session.

In [ ]:
params = {"q": {a: float(Q[a]) for a in ADAPTER_ORDER},
          "tau_learned_f1": float(TAU_learned_f1),
          "tau_learned_1pct_fpr": float(TAU_learned_fpr),
          "tau_or_f1": float(TAU_or_f1),
          "label_ids": {a: LABEL_IDS[a] for a in ADAPTER_ORDER}}
with open(f"{OUT}/fusion_params.json", "w") as f:
    json.dump(params, f, indent=2)
print(json.dumps(params, indent=2))
print("\nSaved: val_scores.parquet, test_scores.parquet, fusion_params.json in", OUT)
# Optionally push to a private HF dataset repo so results survive the session:
# from huggingface_hub import HfApi
# api = HfApi(token=HF_TOKEN)
# api.create_repo('hirushafernando/slm-shield-results', repo_type='dataset', private=True, exist_ok=True)
# api.upload_folder(folder_path=OUT, repo_id='hirushafernando/slm-shield-results', repo_type='dataset')